# Spätere Auswertung hochladen · optional

`config/gasi_analysis.json` beschreibt Ergebnisse, Auswahl und tatsächliche Server-Eingaben.
Der zentrale Sessionmodus gilt auch hier: False zeigt die Vorschau, True schreibt.
Dieses Notebook verwendet vorhandene Messungen und lädt deren Originale nicht erneut hoch.


In [ ]:
from pathlib import Path
import os, sys
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "gasi_session.py").is_file())
os.chdir(repo)
sys.path.insert(0, str(repo))


In [ ]:
from pathlib import Path
import importlib
import gasi_analysis
from IPython.display import FileLink, display

gasi_analysis = importlib.reload(gasi_analysis)
path = Path("config/gasi_analysis.json")
if not path.exists():
    with path.open("xb") as target:
        target.write(Path("config/gasi_analysis.example.json").read_bytes())
config = gasi_analysis.runs.load_run(path)
issues = gasi_analysis.validate_analysis(config)
targets, message = gasi_analysis.ob.load_target_config()
print(message)
try:
    target = gasi_analysis.runs._targets(targets, ("analysis",))["analysis"]
    print("Auswertungsziel:", gasi_analysis.ob.target_path(target))
except gasi_analysis.ob.CheckFailed as exc:
    issues.append(str(exc))
preview_sha256 = None
if issues:
    for issue in issues:
        print("-", issue)
else:
    bundle = gasi_analysis.prepare_analysis(config)
    preview_sha256 = bundle["sha256"]
    print("Lokale Vorschau geprüft; Eingaben und Rechte werden vor dem Schreiben live erneut geprüft.")
    for item in config["inputs"]:
        print("Eingabe:", item["measurement_id"], item["dataset_id"], item["file_path"])
    print("Ergebnisdateien:", len(bundle["entries"]), "; Schreibmodus siehe Session.")


In [ ]:
result, message = gasi_analysis.upload_analysis(
    expected_manifest_sha=globals().get("preview_sha256"))
print(message)
if result:
    print(result)
